In [15]:
import pandas as pd
import re
from google.colab import files

# 1) Excel'i oku
in_path = "/content/sample_data/v1_f_IT_Department_Sorunlar_.xlsx"
df = pd.read_excel(in_path)

# 2) Metin kolonu oluştur (subject boşsa body kullan)
df["text"] = df.get("subject", "").fillna("").astype(str).str.strip()
if "body" in df.columns:
    df.loc[df["text"].eq(""), "text"] = df["body"].fillna("").astype(str)

# 3) Regex desenleri (HATA vs İSTEK)
hata_pat = re.compile(
    r"\b(sorun|hata|atıyor|bekletiyor|izin vermiyor|düzelmedi|yüklenmiyor|yanıt vermiyor|"
    r"çalışmıyor|calismiyor|ulaşılamıyor|ulasilamiyor|problem|arıza|error|exception)\b",
    re.IGNORECASE
)

istek_pat = re.compile(
    r"\b(talep|istek|ekleme|eklenmesi|giriş yetkisi|kurulum|kurulumu|yükleme|"
    r"yukleme|install|request|yeni kullanıcı|yeni kullanici|yetki)\b",
    re.IGNORECASE
)

# Tie-break
hata_strong = re.compile(
    r"\b(çalışmıyor|calismiyor|ulaşılamıyor|ulasilamiyor|hata mesajı|error|exception)\b",
    re.IGNORECASE
)
istek_strong = re.compile(
    r"\b(yeni kullanıcı|yeni kullanici|kurulum|kurulumu|yetki talebi|erişim talebi|vpn erişim isteği)\b",
    re.IGNORECASE
)

def coarse_classify(text: str) -> str:
    t = text or ""
    h = bool(hata_pat.search(t))
    i = bool(istek_pat.search(t))

    if h and not i:
        return "HATA"
    if i and not h:
        return "İSTEK"
    if h and i:
        if len(hata_strong.findall(t)) > len(istek_strong.findall(t)):
            return "HATA"
        if len(istek_strong.findall(t)) > len(hata_strong.findall(t)):
            return "İSTEK"
        return "HATA"
    return "BELİRSİZ"

# 4) Uygula
df["coarse_label"] = df["text"].apply(coarse_classify)

# 5) Özet tablo
summary = (
    df["coarse_label"]
    .value_counts()
    .rename_axis("coarse_label")
    .reset_index(name="count")
)

# 6) Çok sayfalı Excel çıktısı
out_path = "/content/it_helpdesk_regex_on_siniflandirma.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="ALL", index=False)
    summary.to_excel(writer, sheet_name="SUMMARY", index=False)
    df[df["coarse_label"] == "HATA"].to_excel(writer, sheet_name="HATA", index=False)
    df[df["coarse_label"] == "İSTEK"].to_excel(writer, sheet_name="ISTEK", index=False)
    df[df["coarse_label"] == "BELIRSIZ"].to_excel(writer, sheet_name="BELIRSIZ", index=False)

print("Kaydedildi:", out_path)
print(summary)

# 7) Excel dosyasını otomatik indir
files.download(out_path)


Kaydedildi: /content/it_helpdesk_regex_on_siniflandirma.xlsx
  coarse_label  count
0         HATA   2351
1        İSTEK    649


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>